<div align="center">

# Week 01: Foundations: Cyber-Physical Systems, Feedback Loops and the Power Balance

**Cyber-Physical Systems (BIL-435), hands-on Colab notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Week overview](https://github.com/utkukose/cyber-physical-systems-NB-lecture/tree/main/weeks/week-01) · [Lecture page](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-01/lecture.html) · [Interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-01/lab.html) · [PDF notes](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-01/Week01_Lecture_Notes.pdf)

</div>

## How to use this Colab notebook

This Colab notebook is the hands-on part of the week. The concepts are explained on the [lecture page](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-01/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Cells titled *Observe and reflect* ask open questions about what the code shows. They have no solution cell, and a short answer in a text cell of your copy (Insert, Text cell) is enough. Cells titled *Python note* explain a piece of the Python language used nearby. Nothing needs to be installed.

**Starting in the middle.** Each numbered section can also be opened on its own, for example from an orange Colab box of the lecture page. Its first code cell, *Catch-up*, runs the code of the earlier sections without their printouts and figures. When the Colab notebook runs from the top, the catch-up cells notice it and do nothing.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback. It also stores an invented spring weekday of the campus: The hourly load of the buildings, the chargers of the car park and the wind speed. The second cell keeps the expected values of the exercises in the table `ANSWERS`, with its code hidden, so that no answer stands next to its exercise.

In [ ]:
# Environment: Everything used by this course is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import io
import math
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (7, 3.6),
                     "axes.grid": True, "grid.alpha": 0.3})

def check(name, value, expected=None, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback.
    Without an expected value in the call, check takes the value and its tolerance
    from ANSWERS, the table of the next cell."""
    if expected is None:
        answer, tol = ANSWERS[name]
        expected = answer()
    if value is None:
        print(f"{name}: not answered yet. Replace None and run the cell again.")
        return False
    try:
        scale = max(1.0, abs(float(expected)))
        ok = abs(float(value) - float(expected)) <= tol * scale
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else
                         "not yet. Compare your steps with the lecture."))
    return ok

# An invented spring weekday on the campus, one value per hour (kW and m/s)
HOURS = np.arange(24)
LOAD_KW = np.array([262, 255, 250, 248, 252, 270, 330, 430, 520, 570, 585, 590,
                    575, 580, 585, 575, 560, 530, 520, 500, 450, 390, 320, 280], float)
EV_KW = np.array([0, 0, 0, 0, 0, 0, 0, 22, 66, 66, 44, 22,
                  22, 22, 22, 44, 66, 110, 110, 88, 44, 22, 0, 0], float)
WIND_MS = np.array([7.8, 8.0, 8.2, 8.0, 7.6, 7.0, 6.2, 5.4, 4.8, 4.4, 4.2, 4.6,
                    5.0, 5.2, 5.0, 4.6, 4.2, 3.8, 3.6, 3.5, 3.8, 4.6, 5.8, 7.0])
print("Setup complete. Running in Colab:", IN_COLAB, "| hours:", len(HOURS),
      "| peak building load:", LOAD_KW.max(), "kW")

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
ANSWERS = {
    "Exercise 1.1": (lambda: 400 + 250, 1e-9),
    "Exercise 1.2": (lambda: 3, 1e-9),
    "Exercise 1.3": (lambda: 13.5 * 6, 1e-9),
    "Exercise A": (lambda: float(day['import_kwh'].sum()), 0.05),
    "Exercise B": (lambda: 250 * (10 ** 3 - 3 ** 3) / (12 ** 3 - 3 ** 3), 1e-6),
    "Exercise C": (lambda: math.ceil(math.log(10 / 130) / math.log(abs(1 - 0.8))), 1e-9),
    "Exercise D": (lambda: float(STAGE1['peak import with the battery (kW)']), 1e-6),
}

## Python step 1: Values, variables, lists and decisions

Python is the language of all exercises of this course &#91;[1](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-01/lecture.html#ref-1)&#93;. This step introduces the first tools with the campus microgrid. They are values with units stored in variables, lists of devices that a loop walks through, decisions with `if` and functions that give a calculation a name.

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "WIND_MS" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-01/"
               "NB01_cps_foundations.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "python-step":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

### Values, variables and units

In [ ]:
# Python practice: a small example for learning the language
pv_peak_kw = 400          # the solar plant, an integer
battery_kwh = 500.0       # the battery, a float
site = "SDU campus"       # a string
island = False            # a boolean: connected to the city grid
print(site, "| solar plant", pv_peak_kw, "kW | battery", battery_kwh, "kWh")
hours = battery_kwh / 250  # full power of the inverter: 250 kW
print("the battery can deliver 250 kW for", hours, "hours")

A variable is a name for a value, created with `=`. Python does not know units, so the names carry them: `pv_peak_kw` is a power in kW, `battery_kwh` an energy in kWh. Dividing an energy by a power gives a time in hours, which the last line prints.

### Lists of devices and loops

In [ ]:
# Python practice: a small example for learning the language
buildings = ["engineering", "library", "dormitory", "sports hall", "laboratories"]
peaks_kw = [180, 120, 150, 90, 160]
total = 0
for name, peak in zip(buildings, peaks_kw):
    print(f"{name:<13} {peak:>4} kW")
    total = total + peak
print("sum of the peaks:", total, "kW")

A list keeps values in order between square brackets. The `for` loop takes one element after another, and `zip` walks through two lists side by side. The f-string `f"{name:<13}"` writes a value into the text, here left-aligned in 13 characters.

### Decisions with if

In [ ]:
# Python practice: a small example for learning the language
net_kw = 530                      # load minus solar power, now
limit_kw = 400                    # the import that the campus wants to keep
if net_kw > limit_kw:
    print("discharge the battery by", net_kw - limit_kw, "kW")
elif net_kw < 0:
    print("surplus: charge the battery")
else:
    print("no action needed")

`if` runs its indented block only when its condition is true, `elif` tests a further condition and `else` catches the rest. This is the first decision of a controller: Compare a measurement with a limit and act.

### Functions

In [ ]:
# Python practice: a small example for learning the language
def grid_import(load_kw, pv_kw, wind_kw, battery_kw):
    """Power taken from the city grid; negative values are exports."""
    return load_kw - pv_kw - wind_kw - battery_kw

print(grid_import(650, 120, 40, 0), "kW without the battery")
print(grid_import(650, 120, 40, 90), "kW with 90 kW from the battery")

`def` gives a calculation a name and parameters, and `return` hands back its result. The text in triple quotes, the docstring, says what the function does. The same function serves every hour of the day.

**Quick check.** Why does the name `battery_kwh` carry the unit, while the value 500.0 does not?

<details><summary>Answer</summary>

Python stores only the number. The unit lives in the name, so that nobody adds a power in kW to an energy in kWh by mistake.

</details>

### Exercises for Python step 1

**Exercise 1.1.** Store the installed power of the solar plant, 400 kW, plus that of the wind turbine, 250 kW, in `installed_kw`.

In [ ]:
installed_kw = None   # your answer

check("Exercise 1.1", installed_kw)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
installed_kw = 400 + 250

check("Exercise 1.1", installed_kw)

**Exercise 1.2.** Count the buildings in `peaks_kw` whose peak is above 140 kW with a loop and an `if`, and store the count in `n_big`.

In [ ]:
n_big = None   # your answer

check("Exercise 1.2", n_big)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
n_big = 0
for p in peaks_kw:
    if p > 140:
        n_big = n_big + 1

check("Exercise 1.2", n_big)

**Exercise 1.3.** Write a function `energy_kwh(power_kw, hours)` and use it to store the energy of a 13.5 kW load over 6 hours in `e6`.

In [ ]:
e6 = None   # your answer

check("Exercise 1.3", e6)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def energy_kwh(power_kw, hours):
    return power_kw * hours

e6 = energy_kwh(13.5, 6)

check("Exercise 1.3", e6)

## 1. The power balance over a day &#91;[3](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-01/lecture.html#ref-3)&#93;

*Lecture: [Energy and power: The balance of the microgrid](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-01/lecture.html#energy-and-power)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "grid_import" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-01/"
               "NB01_cps_foundations.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-1":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The balance of the microgrid must hold at every moment, and its energies add up over the day. Python computes the import from the city grid hour by hour and the shares of the day that the campus covers itself.

**From the formulas to the code.** The balance $P_{grid} = P_{load} + P_{ev} - P_{pv} - P_{wind} - P_{bat}$ is computed for all hours at once with arrays. The energy $E = \sum_k P_k\,\Delta t$ with $\Delta t = 1$ h is `.sum()`, and the self-sufficiency $SS = 1 - E_{import} / E_{demand}$ is `self_sufficiency(day)`.

In [ ]:
def pv_power(G, T_amb, p_peak=400.0, gamma=-0.004, k_t=0.03):
    """Solar power in kW from the irradiance G (W/m2) and the air temperature (deg C)."""
    T_cell = T_amb + k_t * G
    return np.maximum(0.0, p_peak * G / 1000 * (1 + gamma * (T_cell - 25)))

def wind_power(v, p_rated=250.0, v_in=3.0, v_rated=12.0, v_out=25.0):
    """Cubic power curve between cut-in and rated speed, rated power up to cut-out."""
    v = np.asarray(v, float)
    p = p_rated * (v ** 3 - v_in ** 3) / (v_rated ** 3 - v_in ** 3)
    p = np.where(v < v_in, 0.0, np.where(v < v_rated, p, p_rated))
    return np.where(v >= v_out, 0.0, p)

G = 950 * np.maximum(0, np.sin(np.pi * (HOURS + 0.5 - 6) / 13))    # a clear sky, W/m2
T_AMB = 12 + 7 * np.sin(np.pi * (HOURS - 9) / 12)                    # air, deg C

def make_day(cloud=1.0):
    """The hourly balance of the campus; cloud scales the irradiance (1 = clear sky)."""
    d = pd.DataFrame({"hour": HOURS, "load_kw": LOAD_KW, "ev_kw": EV_KW,
                      "pv_kw": pv_power(cloud * G, T_AMB).round(1), "wind_kw": wind_power(WIND_MS).round(1)})
    d["net_kw"] = d["load_kw"] + d["ev_kw"] - d["pv_kw"] - d["wind_kw"]
    d["import_kwh"] = d["net_kw"].clip(lower=0)          # one hour: kW equals kWh
    d["export_kwh"] = (-d["net_kw"]).clip(lower=0)
    return d

def self_sufficiency(d):
    demand = (d["load_kw"] + d["ev_kw"]).sum()
    return 1 - d["import_kwh"].sum() / demand

day = make_day()

print(day.round(1).to_string(index=False))
print(f"demand {day.load_kw.sum() + day.ev_kw.sum():.0f} kWh | solar {day.pv_kw.sum():.0f} kWh | "
      f"wind {day.wind_kw.sum():.0f} kWh")
print(f"import {day.import_kwh.sum():.0f} kWh | export {day.export_kwh.sum():.0f} kWh | "
      f"self-sufficiency {self_sufficiency(day):.3f} | peak import {day.net_kw.max():.0f} kW")

**Python note.** A pandas table, a `DataFrame`, holds one column per quantity and one row per hour. Arithmetic on whole columns works element by element, so one line computes the balance of all 24 hours. `clip(lower=0)` keeps the positive part, the imports here and the exports in the next line.

**Exercise A.** Store the energy imported from the city grid over the day, in kWh, in `e_import`.

In [ ]:
e_import = None   # your answer

check("Exercise A", e_import)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
e_import = day["import_kwh"].sum()

check("Exercise A", e_import)

Part A of the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-01/lab.html) runs days on the model of the campus with a rule for the battery. Its button *Copy the results for Colab* copies one row per day. Paste the text into `MY_DAYS`; left empty, the variable makes the code use four days of a simulated student.

In [ ]:
# Paste the results of lab Part A between the triple quotes.
MY_DAYS = """
"""

def run_day(day, threshold_kw=450.0, charge_below_kw=330.0, e_kwh=500.0, p_max=250.0,
            eta=0.95, soc0=0.5, soc_min=0.1, soc_max=0.9):
    """Hourly battery rule: discharge above the threshold, charge below charge_below_kw."""
    soc, out = soc0, []
    for net in day["net_kw"]:
        if net > threshold_kw:                        # discharge: power that leaves the battery
            p = min(net - threshold_kw, p_max, (soc - soc_min) * e_kwh * eta)
        elif net < charge_below_kw:                   # charge: power that enters the inverter
            p = -min(charge_below_kw - net, p_max, (soc_max - soc) * e_kwh / eta)
        else:
            p = 0.0
        soc += (-p / eta if p > 0 else -p * eta) / e_kwh
        out.append((p, net - p, soc))
    res = day.copy()
    res["bat_kw"], res["grid_kw"], res["soc"] = zip(*out)
    return res

def summary(run, weather, threshold, res):
    return dict(run=run, weather=weather, threshold_kw=threshold, peak_import_kw=round(res.grid_kw.max(), 1),
                import_kwh=round(res.grid_kw.clip(lower=0).sum()), soc_end=round(res.soc.iloc[-1], 3))

if MY_DAYS.strip():
    runs = pd.read_csv(io.StringIO(MY_DAYS.strip()))
    source = "your own days on the model table"
else:
    plan = [("sunny", 1.0, 600), ("sunny", 1.0, 450), ("cloudy", 0.4, 450), ("cloudy", 0.4, 400)]
    runs = pd.DataFrame([summary(i + 1, w, th, run_day(make_day(c), th)) for i, (w, c, th) in enumerate(plan)])
    source = "a simulated student"
print("Data:", source)
print(runs.to_string(index=False))
held = runs[runs["peak_import_kw"] <= runs["threshold_kw"] + 0.5]
print("days whose peak import stayed at or below the threshold:", list(held["run"]))

**Python note.** The state of charge falls by $P / \eta$ when the battery discharges and rises by $P\,\eta$ when it charges, so 5 percent is lost in each direction. `zip(*out)` turns the list of tuples into three columns, and `summary` keeps one row per day.

In [ ]:
# Figure: the balance of the day
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].stackplot(HOURS, day.pv_kw, day.wind_kw, labels=["solar", "wind"],
                colors=["#F2B33D", "#6FB5E8"], alpha=0.85, step="post")
ax[0].step(HOURS, day.load_kw + day.ev_kw, where="post", color="#1B2430", label="demand")
ax[0].set_xlabel("hour")
ax[0].set_ylabel("power (kW)")
ax[0].legend(fontsize=8, loc="upper left")
ax[1].bar(HOURS + 0.5, day.net_kw, color=np.where(day.net_kw > 0, "#1E6FA8", "#2E8B57"))
ax[1].axhline(0, color="#56616D", lw=1)
ax[1].set_xlabel("hour")
ax[1].set_ylabel("import (+) or export (-) (kW)")
plt.tight_layout()
plt.show()

**Observe and reflect.** At which hours does the campus export, and at which hours does it import most? Which device of the campus could move energy from the first to the second?

## 2. Models of the sources: Sun and wind &#91;[11](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-01/lecture.html#ref-11)&#93;

*Lecture: [Models of the sources: Sun and wind](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-01/lecture.html#models-of-the-sources)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "self_sufficiency" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-01/"
               "NB01_cps_foundations.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-2":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A model turns the weather into power. Python evaluates the model of the solar plant with its temperature correction and the power curve of the wind turbine, and checks both against the worked examples.

**From the formulas to the code.** $P_{pv} = P_{peak}\,(G / 1000)\,(1 + \gamma\,(T_c - 25))$ is `pv_power(G, T_amb)` with the cell temperature $T_c = T_{amb} + k_t G$. The power curve $P_w = P_r (v^3 - v_{in}^3)/(v_r^3 - v_{in}^3)$ is `wind_power(v)`.

In [ ]:
# The worked examples of the lecture
print(f"solar: G = 800 W/m2, T_cell = 45 C -> {400 * 0.8 * (1 - 0.004 * 20):.1f} kW")
print(f"wind: v = 8 m/s -> {float(wind_power(8.0)):.2f} kW")
v = np.linspace(0, 28, 281)
for vv in [2.5, 3, 6, 9, 12, 20, 25]:
    print(f"v = {vv:>4} m/s: {float(wind_power(vv)):6.1f} kW")
G_grid = np.array([200, 400, 600, 800, 1000])
for t_air in [0, 15, 30]:
    print(f"air {t_air:>2} C:", pv_power(G_grid, t_air).round(1), "kW")

**Python note.** `np.where(condition, a, b)` chooses element by element between two arrays, so the three parts of the power curve need no loop. The functions accept a single number or a whole array.

**Exercise B.** Store the power of the wind turbine at 10 m/s in `p10`, computed with the cubic power curve.

In [ ]:
p10 = None   # your answer

check("Exercise B", p10)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
p10 = float(wind_power(10.0))

check("Exercise B", p10)

**Explore.** Move the sliders: The panel computes the power of the solar plant for an irradiance and an air temperature. If the controls do not appear, run the cell again.

In [ ]:
import ipywidgets as widgets

def explore(G=800, T_air=20):
    p = float(pv_power(G, T_air))
    print(f"cell temperature {T_air + 0.03 * G:.1f} C -> solar power {p:.1f} kW "
          f"({100 * p / 400:.1f} percent of the peak)")

widgets.interact(explore, G=widgets.IntSlider(value=800, min=0, max=1100, step=50),
                 T_air=widgets.IntSlider(value=20, min=-10, max=40, step=1));

In [ ]:
# Figure: the power curve of the wind turbine and the solar power against the irradiance
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(v, wind_power(v), color="#1E6FA8")
ax[0].set_xlabel("wind speed (m/s)")
ax[0].set_ylabel("power (kW)")
Gs = np.linspace(0, 1100, 100)
for t_air, c in [(0, "#6FB5E8"), (15, "#1E6FA8"), (30, "#B53F2E")]:
    ax[1].plot(Gs, pv_power(Gs, t_air), color=c, label=f"air {t_air} C")
ax[1].set_xlabel("irradiance G (W/m2)")
ax[1].set_ylabel("solar power (kW)")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Observe and reflect.** Why does the solar plant deliver less on a hot summer noon than on a cold spring noon with the same sunshine? Between which wind speeds does a small error in the forecast matter most?

## 3. Closing the loop: Your controller &#91;[8](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-01/lecture.html#ref-8)&#93;

*Lecture: [Closing the loop: Peak shaving with the battery](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-01/lecture.html#closing-the-loop)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "wind_power" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-01/"
               "NB01_cps_foundations.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-3":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The battery controller measures the import, compares it with a target and changes the power of the battery once a minute. Python runs this loop step by step and shows how the gain decides between a slow, a fast, an oscillating and an unstable loop.

**From the formulas to the code.** One step of the loop, $P_{bat}[k+1] = P_{bat}[k] + K\,(P_{grid}[k] - P_{target})$, is the body of `run_loop`. For a constant demand, the error shrinks as $e[k] = (1 - K)^k\,e[0]$, computed by `error_after(k, K, e0)`.

In [ ]:
def run_loop(net_kw, target_kw, K, steps=12, p_max=250.0):
    """Peak shaving: the battery adds K times the excess import at every step (once a minute)."""
    p_bat, rows = 0.0, []
    for k in range(steps):
        p_grid = net_kw - p_bat                      # sense: the meter at the grid connection
        error = p_grid - target_kw                   # compute: compare with the target
        rows.append((k, p_bat, p_grid, error))
        p_bat = min(p_max, max(-p_max, p_bat + K * error))   # actuate: new set point
    return pd.DataFrame(rows, columns=["step", "p_bat_kw", "p_grid_kw", "error_kw"])

def error_after(k, K, e0):
    return (1 - K) ** k * e0

print(run_loop(530, 400, 0.5).head(6).round(2).to_string(index=False))
for K in [0.2, 0.5, 1.0, 1.5, 2.2]:
    r = run_loop(530, 400, K)
    print(f"K = {K:.1f}: error after 4 steps {r.error_kw[4]:8.2f} kW (formula "
          f"{error_after(4, K, 130):8.2f}), after 11 steps {r.error_kw[11]:9.2f} kW")

**Python note.** `min(p_max, max(-p_max, x))` limits the set point to the range of the inverter, from -250 to 250 kW. As long as this limit is not reached, the loop follows the formula exactly.

**Exercise C.** With $K = 0.8$ and an initial error of 130 kW, after how many steps is the error below 10 kW for the first time? Store the number in `n_steps`.

In [ ]:
n_steps = None   # your answer

check("Exercise C", n_steps)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
n_steps = next(k for k in range(50) if abs(error_after(k, 0.8, 130)) < 10)

check("Exercise C", n_steps)

Part C of the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-01/lab.html) lets you set the gain on the controller of the battery. Paste the copied rows into `MY_GAINS`, and the code compares your observations with the formula.

In [ ]:
# Paste the results of lab Part C between the triple quotes.
MY_GAINS = """
"""
if MY_GAINS.strip():
    tries = pd.read_csv(io.StringIO(MY_GAINS.strip()))
    source = "your own trials on the bench"
else:
    tries = pd.DataFrame({"trial": [1, 2, 3, 4, 5], "gain": [0.3, 1.0, 1.6, 2.1, 0.7],
                          "behaviour": ["slow", "one step", "oscillating", "unstable", "fast"]})
    source = "a simulated student"
def behaviour(K):
    if K <= 0 or K >= 2:
        return "unstable"
    if K == 1:
        return "one step"
    return "oscillating" if K > 1 else ("slow" if K < 0.5 else "fast")
tries["formula"] = [behaviour(k) for k in tries["gain"]]
tries["factor |1 - K|"] = (1 - tries["gain"]).abs().round(2)
print("Data:", source)
print(tries.to_string(index=False))

In [ ]:
# Figure: the import at the grid connection for five gains
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
for K, c in [(0.3, "#9DB7CC"), (0.5, "#1E6FA8"), (1.0, "#2E8B57"), (1.6, "#C98510"), (2.1, "#B53F2E")]:
    r = run_loop(530, 400, K, steps=10)
    ax[0].plot(r.step, r.p_grid_kw, "o-", color=c, ms=3, label=f"K = {K}")
ax[0].axhline(400, ls=":", color="#56616D")
ax[0].set_xlabel("step (minutes)")
ax[0].set_ylabel("import (kW)")
ax[0].set_ylim(200, 700)
ax[0].legend(fontsize=7)
Ks = np.linspace(0, 2.4, 200)
ax[1].plot(Ks, np.abs(1 - Ks), color="#1E6FA8")
ax[1].axhline(1, ls=":", color="#B53F2E")
ax[1].set_xlabel("gain K")
ax[1].set_ylabel("|1 - K|, the factor per step")
plt.tight_layout()
plt.show()

**Observe and reflect.** Which gain would you choose if the meter were noisy, and which if the load jumped often? Why is a gain of exactly 1 rarely used in practice?

## 4. The campus microgrid, stage 1

*Lecture: [The campus microgrid, stage 1: The power balance of the campus](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-01/lecture.html#the-campus-microgrid-stage-1)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "run_loop" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-01/"
               "NB01_cps_foundations.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-4":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Stage 1 runs the day of Section 1 twice. The first run has no battery, and the second uses a simple rule that discharges the battery above an import threshold and charges it from the surplus and at night. Python compares the energy imported, the peak import and the state of charge.

In [ ]:
base = run_day(day, threshold_kw=10 ** 6, charge_below_kw=-10 ** 6)
shaved = run_day(day)
STAGE1 = {
    "demand (kWh)": round(float((day.load_kw + day.ev_kw).sum())),
    "solar (kWh)": round(float(day.pv_kw.sum())),
    "wind (kWh)": round(float(day.wind_kw.sum())),
    "import without the battery (kWh)": round(float(base.grid_kw.clip(lower=0).sum())),
    "peak import without the battery (kW)": round(float(base.grid_kw.max()), 1),
    "self-sufficiency without the battery": round(float(self_sufficiency(day)), 3),
    "import with the battery (kWh)": round(float(shaved.grid_kw.clip(lower=0).sum())),
    "peak import with the battery (kW)": round(float(shaved.grid_kw.max()), 1),
    "lowest state of charge": round(float(shaved.soc.min()), 3),
    "state of charge at midnight": round(float(shaved.soc.iloc[-1]), 3),
}
for key, value in STAGE1.items():
    print(f"{key:<40} {value}")

**Python note.** `run_day` comes from Section 1. With a threshold of a million kW and a charging limit of minus a million, the battery never acts, which gives the day without the battery.

In [ ]:
# Figure: the import with and without the battery and the state of charge
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].step(HOURS, base.grid_kw, where="post", color="#56616D", label="without the battery")
ax[0].step(HOURS, shaved.grid_kw, where="post", color="#1E6FA8", lw=2, label="with the battery rule")
ax[0].axhline(450, ls=":", color="#B53F2E", label="threshold 450 kW")
ax[0].set_xlabel("hour")
ax[0].set_ylabel("import (kW)")
ax[0].legend(fontsize=8)
ax[1].step(HOURS, 100 * shaved.soc, where="post", color="#2E8B57")
ax[1].axhline(10, ls=":", color="#B53F2E")
ax[1].set_xlabel("hour")
ax[1].set_ylabel("state of charge (%)")
plt.tight_layout()
plt.show()

**Exercise D.** Store the peak import with the battery, as kept in `STAGE1`, in `peak_d`.

In [ ]:
peak_d = None   # your answer

check("Exercise D", peak_d)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
peak_d = STAGE1["peak import with the battery (kW)"]

check("Exercise D", peak_d)

**Observe and reflect.** Why does the import with the battery not stay below 450 kW in every hour? Which setting of the rule would you change first, and what would it cost?

## 5. Your application: The application switch

*Lecture: [Your application](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-01/lecture.html#your-application)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "STAGE1" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-01/"
               "NB01_cps_foundations.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-5":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Every cyber-physical system that exchanges power or material with its surroundings has a balance. The functions of this week compute the balance of other sites after one change of a name.

The sites and their numbers are illustrative. Set `APP` to one of the six names and run the cell.

In [ ]:
APP = "factory"
# choose: campus, factory, hospital, farm, data_centre, ferry_port
APPS = {
    "campus": (LOAD_KW + EV_KW, 400, 250, 500),
    "factory": (np.where((HOURS >= 6) & (HOURS < 22), 900.0, 250.0), 600, 0, 800),
    "hospital": (380 + 120 * np.sin(np.pi * (HOURS - 6) / 12).clip(0), 150, 0, 300),
    "farm": (40 + 60 * ((HOURS >= 5) & (HOURS < 8)) + 50 * ((HOURS >= 17) & (HOURS < 20)), 120, 100, 200),
    "data_centre": (np.full(24, 700.0), 300, 0, 1000),
    "ferry_port": (150 + 900 * np.isin(HOURS, [8, 12, 16, 20]), 250, 250, 1500)}
demand, pv_peak, wind_rated, batt_kwh = APPS[APP]
pv = pv_power(G, T_AMB, p_peak=pv_peak)
wind = wind_power(WIND_MS, p_rated=wind_rated) if wind_rated else np.zeros(24)
site = pd.DataFrame({"hour": HOURS, "load_kw": demand, "ev_kw": 0.0, "pv_kw": pv, "wind_kw": wind})
site["net_kw"] = site.load_kw - site.pv_kw - site.wind_kw
site["import_kwh"] = site.net_kw.clip(lower=0)
site["export_kwh"] = (-site.net_kw).clip(lower=0)
res = run_day(site, threshold_kw=0.85 * site.net_kw.max(), charge_below_kw=0.5 * site.net_kw.max(),
              e_kwh=batt_kwh, p_max=batt_kwh / 2)
print(f"{APP}: demand {site.load_kw.sum():.0f} kWh, local {pv.sum() + wind.sum():.0f} kWh, "
      f"self-sufficiency {self_sufficiency(site):.3f}")
print(f"  peak import {site.net_kw.max():.0f} kW without and {res.grid_kw.max():.0f} kW with a battery of "
      f"{batt_kwh} kWh")

**Python note.** Each entry holds the hourly demand, the solar and wind capacities and the size of a battery. The rule of stage 1 is reused: The battery discharges above 85 percent of the largest net load of the site and charges below 50 percent of it.

**Observe and reflect.** Which site gains most from its battery, and which from more solar panels? What limits the self-sufficiency of a site that runs all night?

## Application challenges

Each challenge transfers the ideas of the week to a field of computing. Choose the one closest to your interests and solve it in a copy of this Colab notebook.

| Field | Challenge |
|---|---|
| Software engineering | Write the balance of the campus as a class with methods for each device and test it with assertions on three hours of the day. |
| Embedded systems | List the sensors, the controller and the actuators of the loop that keeps a refrigerator cold. How often must it sample, and why? |
| Internet of Things | Design a household with solar panels and a battery, and compute its self-sufficiency for the day of Section 1 scaled to 5 kW of solar power. |
| Computer networks | Estimate how many measurements per day the meters of the campus would send if each of its 20 meters reported every 15 minutes, and every second. |
| Control engineering | Replace the fixed gain of the loop by a gain that halves after each step. Does the loop still reach the target, and how fast? |
| Data science | Shift the solar profile by one hour to the left and to the right and measure how the import and the self-sufficiency change. |
| Cybersecurity | List what an attacker could do to the loop of the battery by changing the measurement of the import meter. Which check would you add? |
| Artificial intelligence | A model predicts the load of the next hour with an error of 10 percent. How would you use the prediction in the battery rule of stage 1? |

## After the Colab notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-01/lab.html), tabs *Check yourself* and *Reflect and export*. The weekly task and the research assignment are on the [week overview](https://github.com/utkukose/cyber-physical-systems-NB-lecture/tree/main/weeks/week-01).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>